In [1]:
# ### Required to copy files into google colab. Skip while running
# ! pip install -q kaggle
# from google.colab import files
# files.upload()
# ! mkdir ~/.kaggle
# ! cp kaggle.json ~/.kaggle/
# ! chmod 600 ~/.kaggle/kaggle.json
# ! kaggle datasets download -d nikhilmishradev/zindiweekendfoodrecommendation
# !unzip zindiweekendfoodrecommendation.zip
# !cp weekend_food_recommendation/* ./
# !pip install -q geopandas

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import *
from sklearn.model_selection import *
from sklearn.preprocessing import *
import lightgbm as lgb
from lightgbm import LGBMClassifier
import gc
from sklearn.metrics import f1_score

## Helper library to visualize geographical data
# import geopandas as gpd

## Multiprocessing
from joblib import Parallel, delayed
import json
      
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('max_columns', 100)

In [3]:
train_customers = pd.read_csv('train_customers.csv')
test_customers = pd.read_csv('test_customers.csv')
train_locations = pd.read_csv('train_locations.csv')
test_locations = pd.read_csv('test_locations.csv')
orders = pd.read_csv('orders.csv')
vendors = pd.read_csv('vendors.csv')
sample_sub = pd.read_csv('SampleSubmission.csv')

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:2718: DtypeWarning: Columns (15,16,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
### Customers Data Cleaning and preprocessing

### Removing duplicate customer_ids
test_customers = test_customers.iloc[test_customers['akeed_customer_id'].drop_duplicates().index].reset_index(drop=True)
train_customers = train_customers.iloc[train_customers['akeed_customer_id'].drop_duplicates().index].reset_index(drop=True)

df_customers = pd.concat([train_customers, test_customers], axis=0).reset_index(drop=True)
df_customers['created_at'] = pd.to_datetime(df_customers['created_at'], format='%Y-%m-%d %H:%M:%S')
min_date = df_customers['created_at'].min()
df_customers['updated_at'] = pd.to_datetime(df_customers['updated_at'], format='%Y-%m-%d %H:%M:%S')

for dcol in ['created_at', 'updated_at']:
    for attr in ['month', 'dayofweek']:
        df_customers[dcol + '_' + attr] = getattr(df_customers[dcol].dt, attr)
df_customers['created_at_days_since_start_cust'] = (df_customers['created_at'] - min_date).dt.days
df_customers['updated_at_days_since_start_cust'] = (df_customers['updated_at'] - min_date).dt.days
df_customers['created_at_months_since_start_cust'] = df_customers['created_at_days_since_start_cust']//30
df_customers['n_customers_created_on_month_cust'] = df_customers['created_at_months_since_start_cust'].map(df_customers['created_at_months_since_start_cust'].value_counts())
df_customers['n_customers_created_on_day'] = df_customers['created_at_days_since_start_cust'].map(df_customers['created_at_days_since_start_cust'].value_counts())


# df_customers['n_customers_created_on_prev_day'] = df_customers['created_at_days_since_start'].map(df_customers['created_at_days_since_start'].value_counts().sort_index().shift(1))
# df_customers['n_customers_created_on_next_day'] = df_customers['created_at_days_since_start'].map(df_customers['created_at_days_since_start'].value_counts().sort_index().shift(-1))
# df_customers['n_customers_updated_on_prev_day'] = df_customers['updated_at_days_since_start'].map(df_customers['updated_at_days_since_start'].value_counts().sort_index().shift(1))
# df_customers['n_customers_updated_on_next_day'] = df_customers['updated_at_days_since_start'].map(df_customers['updated_at_days_since_start'].value_counts().sort_index().shift(-1))


df_customers['updation_creation_diff'] = (df_customers['updated_at'] - df_customers['created_at']).dt.seconds
df_customers[['language', 'gender']] = df_customers[['language', 'gender']].apply(lambda x: pd.factorize(x)[0])

df_customers = df_customers.rename({'akeed_customer_id': 'customer_id'}, axis=1)

# tmp = df_customers.sort_values(by='updated_at')
# df_customers['prev_customer_update_diff'] = ((tmp['updated_at'] - tmp['updated_at'].shift(1)).sort_index()).dt.seconds
# df_customers['next_customer_update_diff'] = ((tmp['updated_at'] - tmp['updated_at'].shift(-1)).sort_index()).dt.seconds
# df_customers['prev_customer_update_dss'] = tmp['updated_at_days_since_start_cust'].shift(1).sort_index()
# df_customers['next_customer_update_dss'] = tmp['updated_at_days_since_start_cust'].shift(-1).sort_index()


# tmp = df_customers.sort_values(by='created_at')
# df_customers['prev_customer_created_diff'] = ((tmp['created_at'] - tmp['created_at'].shift(1)).sort_index()).dt.seconds
# df_customers['next_customer_created_diff'] = ((tmp['created_at'] - tmp['created_at'].shift(-1)).sort_index()).dt.seconds
# df_customers['prev_customer_create_dss'] = tmp['created_at_days_since_start_cust'].shift(1).sort_index()
# df_customers['next_customer_create_dss'] = tmp['created_at_days_since_start_cust'].shift(-1).sort_index()


df_customers = df_customers.drop(['created_at', 'updated_at',  'dob'], axis=1).fillna(-1)
train_customers, test_customers = df_customers[:train_customers.shape[0]].reset_index(drop=True), df_customers[train_customers.shape[0]:].reset_index(drop=True)

print(train_customers.shape, test_customers.shape)
train_customers =  train_customers.drop_duplicates()
test_customers = test_customers.drop_duplicates()
train_customers.shape, test_customers.shape

(34523, 15) (9753, 15)


((34523, 15), (9753, 15))

In [5]:
%%time
trn_unq = train_locations['location_number'].unique()
tst_unq = test_locations['location_number'].unique()

all_locations = pd.concat([train_locations, test_locations], axis=0)
fltr = all_locations['location_number'].isin(tst_unq)
all_locations = all_locations[fltr]
cust_locs = all_locations.groupby('customer_id')['location_number'].unique()

mlb = MultiLabelBinarizer()
op = pd.DataFrame(mlb.fit_transform(cust_locs)).iloc[:, 1:]
op.columns = [f'loc_{l}_visited_by_customer' for l in op.columns]
op['customer_id'] = cust_locs.index.values
op.head(2)

train_locations = pd.merge(train_locations, op, on = 'customer_id', how = 'left')
test_locations = pd.merge(test_locations, op, on = 'customer_id', how = 'left')
train_locations.shape, test_locations.shape

CPU times: user 5.86 s, sys: 98.9 ms, total: 5.96 s
Wall time: 5.88 s


In [6]:
del all_locations, op, trn_unq, tst_unq, cust_locs
gc.collect()

13

In [7]:
train_locations['n_locations_for_customer'] = train_locations['customer_id'].map(train_locations.groupby('customer_id').size())
test_locations['n_locations_for_customer'] = test_locations['customer_id'].map(test_locations.groupby('customer_id').size())

train_locations['n_unqiue_locations_for_customer'] = train_locations['customer_id'].map(train_locations.groupby('customer_id')['location_number'].nunique())
test_locations['n_unqiue_locations_for_customer'] = test_locations['customer_id'].map(test_locations.groupby('customer_id')['location_number'].nunique())

location_number_counts_mapper = train_locations['location_number'].value_counts() + test_locations['location_number'].value_counts()
train_locations['location_number_count'] = train_locations['location_number'].map(location_number_counts_mapper)
test_locations['location_number_count'] = test_locations['location_number'].map(location_number_counts_mapper)

In [8]:
%%time

## To replace outliers by only their decimal parts
for df_tmp in [train_locations, test_locations, vendors]:
  for col in ['latitude', 'longitude']:
    fltr = (df_tmp[col] < -3) | (df_tmp[col] > 3)
    df_tmp.loc[fltr, col] = df_tmp.loc[fltr, col].apply(lambda x: x - int(x)) ## To get only the decimal part with sign, like -3.14 - (3) = -0.14

for df_tmp in [train_locations, test_locations]:
  for col in ['latitude', 'longitude']:
    df_tmp[col +'_std_for_customer'] = df_tmp['customer_id'].map(df_tmp.groupby('customer_id')[col].std())
    df_tmp[col +'_mean_for_customer'] = df_tmp['customer_id'].map(df_tmp.groupby('customer_id')[col].mean())
    df_tmp[col +'_min_for_customer'] = df_tmp['customer_id'].map(df_tmp.groupby('customer_id')[col].min())
    df_tmp[col +'_max_for_customer'] = df_tmp['customer_id'].map(df_tmp.groupby('customer_id')[col].max())
    df_tmp[col + '_range_for_customer'] = df_tmp[col +'_max_for_customer'] - df_tmp[col +'_min_for_customer']
    

mapper = train_locations.groupby('location_number')['customer_id'].nunique() + test_locations.groupby('location_number')['customer_id'].nunique()
train_locations['num_customers_in_location'] = train_locations['location_number'].map(mapper)
test_locations['num_customers_in_location'] = test_locations['location_number'].map(mapper)

CPU times: user 577 ms, sys: 7.06 ms, total: 584 ms
Wall time: 581 ms


In [9]:
del mapper
gc.collect()

13

In [10]:
### Cleaning and Preprocessing location information
print("\nBefore dropping duplicates, shape\n")
test_locations.shape, train_locations.shape
train_locations = train_locations.drop_duplicates()
test_locations = test_locations.drop_duplicates()
print("\nAfter dropping duplicates, shape\n")
test_locations.shape, train_locations.shape

##### almost half of the values are NaN
print('\nNaN values % is', end = '')
train_locations['location_type'].isnull().sum()/train_locations.shape[0], test_locations['location_type'].isnull().sum()/test_locations.shape[0]

### Filling NaN values with Unknown token
train_locations['location_type'] = train_locations['location_type'].fillna('UNK') # Unknown
test_locations['location_type'] = test_locations['location_type'].fillna('UNK')

### Converting 'location_type' to a number
mapper_location_type = {'UNK': -1, 'Home': 0, 'Work': 1, 'Other': 2}
train_locations['location_type'] = train_locations['location_type'].map(mapper_location_type)
test_locations['location_type'] = test_locations['location_type'].map(mapper_location_type)

### New values
print('\nNaN values % is', end = ' ', )
train_locations['location_type'].isnull().sum()/train_locations.shape[0], test_locations['location_type'].isnull().sum()/test_locations.shape[0]


Before dropping duplicates, shape



((16720, 30), (59503, 30))


After dropping duplicates, shape



((16720, 30), (59503, 30))


NaN values % is

(0.45727106196326234, 0.45753588516746413)


NaN values % is 

(0.0, 0.0)

In [11]:
### Merging Locations and Customers table

train_locations = pd.merge(train_locations, train_customers, on = 'customer_id', how = 'left')
test_locations = pd.merge(test_locations, test_customers, on = 'customer_id', how = 'left')
train_locations.shape, test_locations.shape

### Renaming to differentiate with vendor columns of the same name
train_locations = train_locations.rename({'latitude': 'cust_latitude', 'longitude': 'cust_longitude'}, axis=1)
test_locations = test_locations.rename({'latitude': 'cust_latitude', 'longitude': 'cust_longitude'}, axis=1)

### Ensuring there are no duplicate
test_locations = test_locations.drop_duplicates()
train_locatioons = train_locations.drop_duplicates()
train_locations.shape, test_locations.shape

((59503, 44), (16720, 44))

((59503, 44), (16720, 44))

In [12]:
all_locations = pd.concat([train_locations, test_locations], axis=0).reset_index(drop=True)
print(all_locations.shape)

n_cust_per_loc_created_on_day = all_locations.groupby(['created_at_days_since_start_cust', 'location_number']).size().rename('n_cust_per_loc_created_on_day_size')
all_locations = pd.merge(all_locations, n_cust_per_loc_created_on_day, on = ['created_at_days_since_start_cust', 'location_number'], how = 'left')

n_cust_per_loc_updated_on_day = all_locations.groupby(['updated_at_days_since_start_cust', 'location_number']).size().rename('n_cust_per_loc_updated_on_day_size')
all_locations = pd.merge(all_locations, n_cust_per_loc_updated_on_day, on = ['updated_at_days_since_start_cust', 'location_number'], how = 'left')

train_locations, test_locations = all_locations[:train_locations.shape[0]].reset_index(drop=True), all_locations[train_locations.shape[0]:].reset_index(drop=True)
train_locations.shape, test_locations.shape

(76223, 44)


((59503, 46), (16720, 46))

In [13]:
### Vendors data cleaning and preprocessing

vendors['created_at'] = pd.to_datetime(vendors['created_at'], format='%Y-%m-%d %H:%M:%S')
vendors['updated_at'] = pd.to_datetime(vendors['updated_at'], format='%Y-%m-%d %H:%M:%S')
for dcol in ['created_at', 'updated_at']:
    for attr in ['month', 'dayofweek']:
        vendors['vendor_' + dcol + '_' + attr] = getattr(vendors[dcol].dt, attr)
vendors['updation_creation_diff'] = (vendors['updated_at'] - vendors['created_at']).dt.seconds
vendors['created_at_days_since_start_ven'] = (vendors['created_at'] - min_date).dt.days
vendors['updated_at_days_since_start_ven'] = (vendors['updated_at'] - min_date).dt.days


vendors = vendors.rename({'latitude': 'vendor_latitude', 'longitude': 'vendor_longitude', 'created_at': 'vendor_created_at', 
                          'updated_at': 'vendor_updated_at', 'updation_creation_diff': 'vendor_updation_creation_diff'}, axis=1)


cat_cols = ['is_akeed_delivering', 'vendor_category_en']
vendors[cat_cols] = vendors[cat_cols].apply(lambda x: pd.factorize(x)[0])


### Vendors feature engineering
time_cols = [c for c in vendors.columns if 'from_time' in c or 'to_time' in c]
# print(time_cols)
vendors[time_cols] = vendors[time_cols].apply(lambda x: pd.to_datetime(x, errors='coerce'))
time_cols_1 = [c for c in time_cols if '1' in c]
time_cols_2 = [c for c in time_cols if '2' in c]
# print(time_cols_1)

for wday in ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']:
    wday_cols = [c for c in time_cols_1 if wday in c]
    from_col, to_col = wday_cols[0], wday_cols[1]
    vendors[wday + 'hour_start'] = vendors[from_col].dt.hour 
    vendors[wday + 'hour_end'] = vendors[to_col].dt.hour
    vendors[wday + '_num_hours_open'] = vendors[wday + 'hour_end'] - vendors[wday + 'hour_start']
    # vendors = vendors.drop([wday + 'hour_start', wday + 'hour_end'], axis = 1)

vendors.head(2)

,id,authentication_id,vendor_latitude,vendor_longitude,vendor_category_en,vendor_category_id,delivery_charge,serving_distance,is_open,OpeningTime,OpeningTime2,prepration_time,commission,is_akeed_delivering,discount_percentage,status,verified,rank,language,vendor_rating,sunday_from_time1,sunday_to_time1,sunday_from_time2,sunday_to_time2,monday_from_time1,monday_to_time1,monday_from_time2,monday_to_time2,tuesday_from_time1,tuesday_to_time1,tuesday_from_time2,tuesday_to_time2,wednesday_from_time1,wednesday_to_time1,wednesday_from_time2,wednesday_to_time2,thursday_from_time1,thursday_to_time1,thursday_from_time2,thursday_to_time2,friday_from_time1,friday_to_time1,friday_from_time2,friday_to_time2,saturday_from_time1,saturday_to_time1,saturday_from_time2,saturday_to_time2,primary_tags,open_close_flags,vendor_tag,vendor_tag_name,one_click_vendor,country_id,city_id,vendor_created_at,vendor_updated_at,device_type,display_orders,vendor_created_at_month,vendor_created_at_dayofweek,vendor_updated_at_month,vendor_updated_at_dayofweek,vendor_updation_creation_diff,created_at_days_since_start_ven,updated_at_days_since_start_ven,sundayhour_start,sundayhour_end,sunday_num_hours_open,mondayhour_start,mondayhour_end,monday_num_hours_open,tuesdayhour_start,tuesdayhour_end,tuesday_num_hours_open,wednesdayhour_start,wednesdayhour_end,wednesday_num_hours_open,thursdayhour_start,thursdayhour_end,thursday_num_hours_open,fridayhour_start,fridayhour_end,friday_num_hours_open,saturdayhour_start,saturdayhour_end,saturday_num_hours_open
0,4,118597.0,-0.588596,0.754434,0,2.0,0.0,6.0,1.0,11:00AM-11:30PM,-,15,0.0,0,0.0,1.0,1,11,EN,4.4,2020-08-20,2020-08-20 00:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 10:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 00:30:00,2020-08-20 10:00:00,2020-08-20 23:59:00,"{""primary_tags"":""4""}",1.0,"2,4,5,8,91,22,12,24,16,23","Arabic,Breakfast,Burgers,Desserts,Free Deliver...",Y,1.0,1.0,2018-01-30 14:42:04,2020-04-07 15:12:43,3,1,1,1,4,1,1839,-9,789,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13,118608.0,-0.471654,0.744470,0,2.0,0.7,5.0,1.0,08:30AM-10:30PM,-,14,0.0,0,0.0,1.0,1,11,EN,4.7,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 19:30:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 19:30:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,2020-08-20,2020-08-20 01:30:00,2020-08-20 08:00:00,2020-08-20 23:59:00,"{""primary_tags"":""7""}",1.0,"4,41,51,34,27,15,24,16,28","Breakfast,Cakes,Crepes,Italian,Pasta,Pizzas,Sa...",Y,1.0,1.0,2018-05-03 12:32:06,2020-04-05 20:46:03,3,1,5,3,4,6,29637,84,788,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0


In [14]:
vendors['primary_tags'] = vendors['primary_tags'].fillna('{"primary_tags": "-1"}')
vendors['primary_tags'] = vendors['primary_tags'].apply(lambda x: json.loads(x)['primary_tags']).astype('int')

from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(binary=True, use_idf = False, norm = False)
vendor_tags = vec.fit_transform(vendors['vendor_tag_name'].fillna("No tag")).toarray()
vendor_tags = pd.DataFrame(vendor_tags).astype('int')
vendor_tags.columns = [f'vendor_tag_{i}' for i in range(vendor_tags.shape[1])]
freq_tags = vendor_tags.sum(axis=0)[vendor_tags.sum(axis=0) > 20].index.tolist()
print(len(freq_tags))
vendor_tags = vendor_tags[freq_tags]
vendors = pd.concat([vendors, vendor_tags], axis = 1)

8


In [15]:
### Removing datetime and object columns
''' Later datetime columns can be used to add more features '''

to_drop_cols = vendors.select_dtypes(['object', 'datetime64[ns]']).columns.tolist()
print(to_drop_cols)
vendors = vendors.drop(to_drop_cols + ['authentication_id'], axis=1)
vendors = vendors.rename({'id': 'vendor_id'}, axis=1)

' Later datetime columns can be used to add more features '

['OpeningTime', 'OpeningTime2', 'language', 'sunday_from_time1', 'sunday_to_time1', 'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1', 'monday_to_time1', 'monday_from_time2', 'monday_to_time2', 'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2', 'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1', 'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1', 'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2', 'friday_from_time1', 'friday_to_time1', 'friday_from_time2', 'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1', 'saturday_from_time2', 'saturday_to_time2', 'vendor_tag', 'vendor_tag_name', 'one_click_vendor', 'vendor_created_at', 'vendor_updated_at']


In [16]:
mlb = MultiLabelBinarizer()
vendors_all_locations = orders.groupby('vendor_id')['LOCATION_NUMBER'].unique().reset_index()
for i in range(24):
  vendors_all_locations[f'location_{i}_delivered_by_resturant'] = 0
sel_cols = [f'location_{i}_delivered_by_resturant' for i in range(24)]
vendors_all_locations[sel_cols] = mlb.fit_transform(vendors_all_locations['LOCATION_NUMBER'])
use_cols = [c for c in sel_cols if (vendors_all_locations[c].std() != 0) and (vendors_all_locations[c].mean() > 0.01) and (vendors_all_locations[c].mean() < 0.96)]
vendors_all_locations = vendors_all_locations[['vendor_id'] + use_cols]
vendors_all_locations.head(2)

,vendor_id,location_6_delivered_by_resturant,location_7_delivered_by_resturant,location_8_delivered_by_resturant,location_9_delivered_by_resturant,location_10_delivered_by_resturant,location_11_delivered_by_resturant,location_12_delivered_by_resturant,location_13_delivered_by_resturant,location_14_delivered_by_resturant,location_15_delivered_by_resturant,location_16_delivered_by_resturant,location_17_delivered_by_resturant,location_18_delivered_by_resturant
0,4,1,1,1,1,0,0,0,0,0,0,0,0,0
1,13,1,1,1,0,0,0,0,0,0,0,0,0,0


In [17]:
orders['promo_code'] = (~orders['promo_code'].isnull())*1
cols = ['vendor_rating', 'deliverydistance', 'preparationtime', 'item_count']
for c in cols:
  orders[c] = orders.groupby('vendor_id')[c].transform(lambda x: x.fillna(x.mean()))


orders['vendor_rating'] = orders.groupby('vendor_id')['vendor_rating'].transform(lambda x: x.fillna(x.mean()))
orders['discount_percent'] = orders['vendor_discount_amount']/orders['grand_total']
orders['is_favorite'] = orders['is_favorite'].map({'Yes': 1, 'No': 0}).astype('float')
orders['is_rated'] = orders['is_rated'].map({'Yes': 1, 'No': 0}).astype('float')
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'], format='%Y-%m-%d %H:%M:%S')
orders['created_at'] = pd.to_datetime(orders['created_at'], format='%Y-%m-%d %H:%M:%S')
orders['delivery_date_days_since_start'] = (orders['delivery_date'] - orders['delivery_date'].min()).dt.days
orders['diff_delivery_created'] = (orders['delivery_date'] - orders['created_at']).dt.seconds
orders = pd.merge(orders, orders.groupby(['vendor_id', 'delivery_date_days_since_start']).size().rename('n_deliveries_on_day'), on = ['vendor_id', 'delivery_date_days_since_start'],
                  how = 'left')

In [18]:
#### TODO: Aggregation of time based columns
agg_dict = {'customer_id': ['nunique'],
           'item_count': ['mean', 'min', 'max'],
           'grand_total': ['mean', 'min', 'max'],
            'payment_mode': ['nunique'],
            'vendor_discount_amount': ['mean'],
            'discount_percent': ['mean', 'std', 'min', 'max'],
            'driver_rating': ['mean', 'std', 'min', 'max'],
            'deliverydistance': ['mean', 'std', 'min', 'max'],
            'preparationtime': ['mean', 'std', 'min', 'max'],
            'is_favorite': ['mean', 'std'], ### can be tuned
            'is_rated': ['mean', 'std'],
            'akeed_order_id': ['size'],
           'n_deliveries_on_day': ['min', 'max', 'mean'],
            'LOCATION_NUMBER': ['nunique'],
            'delivery_date_days_since_start': ['min', 'max', 'median'],
            'diff_delivery_created': ['min', 'max', 'median'],
            'vendor_rating': ['mean', 'std'],
            'promo_code': ['mean'],
           }
orders_grouped_by_vendor_id = orders.groupby('vendor_id').agg(agg_dict)

for name, df_tmp in [('vendor_id', orders_grouped_by_vendor_id)]:
    df_tmp.columns = ['_'.join(c).strip('_') + '_grpd_by_' + name for c in df_tmp.columns]

In [19]:
### Adding aggregated location informtation using orders table
### TODO: Time Features

train_locations = train_locations.rename({'location_number': 'LOCATION_NUMBER'}, axis=1)
test_locations = test_locations.rename({'location_number': 'LOCATION_NUMBER'}, axis=1)

print(f'Old number of columns in vendor dataframe : {vendors.shape[1]}')
vendors = vendors.rename({'location_number': 'LOCATION_NUMBER'}, axis=1)
# vendors = pd.merge(vendors, orders_grouped_by_vendor_id, on = 'vendor_id', how = 'left')
print(f'New number of columns in vendor dataframe : {vendors.shape[1]}')
vendors.head(2)

Old number of columns in vendor dataframe : 58
New number of columns in vendor dataframe : 58


,vendor_id,vendor_latitude,vendor_longitude,vendor_category_en,vendor_category_id,delivery_charge,serving_distance,is_open,prepration_time,commission,is_akeed_delivering,discount_percentage,status,verified,rank,vendor_rating,primary_tags,open_close_flags,country_id,city_id,device_type,display_orders,vendor_created_at_month,vendor_created_at_dayofweek,vendor_updated_at_month,vendor_updated_at_dayofweek,vendor_updation_creation_diff,created_at_days_since_start_ven,updated_at_days_since_start_ven,sundayhour_start,sundayhour_end,sunday_num_hours_open,mondayhour_start,mondayhour_end,monday_num_hours_open,tuesdayhour_start,tuesdayhour_end,tuesday_num_hours_open,wednesdayhour_start,wednesdayhour_end,wednesday_num_hours_open,thursdayhour_start,thursdayhour_end,thursday_num_hours_open,fridayhour_start,fridayhour_end,friday_num_hours_open,saturdayhour_start,saturdayhour_end,saturday_num_hours_open,vendor_tag_0,vendor_tag_5,vendor_tag_6,vendor_tag_16,vendor_tag_17,vendor_tag_24,vendor_tag_60,vendor_tag_61
0,4,-0.588596,0.754434,0,2.0,0.0,6.0,1.0,15,0.0,0,0.0,1.0,1,11,4.4,4,1.0,1.0,1.0,3,1,1,1,4,1,1839,-9,789,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,1,1,1,1,1,1
1,13,-0.471654,0.744470,0,2.0,0.7,5.0,1.0,14,0.0,0,0.0,1.0,1,11,4.7,7,1.0,1.0,1.0,3,1,5,3,4,6,29637,84,788,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0,1,1,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0,1,0,0,0,0,1,1


In [20]:
%%time
## Cross Product
train_locations['key'] = 0
test_locations['key'] = 0
vendors['key'] = 0

train = pd.merge(train_locations, vendors, on = 'key')
test = pd.merge(test_locations, vendors, on = 'key')


### Adding ID column to both train and test

icol = 'CID X LOC_NUM X VENDOR'
train['CID X LOC_NUM X VENDOR'] = train['customer_id'].astype('str') + ' X ' + train['LOCATION_NUMBER'].astype('str') + ' X ' + train['vendor_id'].astype('str')
test['CID X LOC_NUM X VENDOR'] = test['customer_id'].astype('str') + ' X ' + test['LOCATION_NUMBER'].astype('str') + ' X ' + test['vendor_id'].astype('str')

train['target'] =  train[icol].isin(orders[icol]) * 1
test['target'] = test[icol].isin(orders[icol])

### Removing columns which only have one unique value

to_drop_cols = [c for c in train.columns if train[c].nunique() == 1] + [icol]
train = train.drop(to_drop_cols, axis=1)
test = test.drop(to_drop_cols, axis=1)
train.shape, test.shape

gc.collect()

CPU times: user 32.7 s, sys: 5.09 s, total: 37.8 s
Wall time: 37.8 s


In [21]:

for df_tmp in [train, test]:
  df_tmp['lat_diff'] = np.abs(df_tmp['cust_latitude'] - df_tmp['vendor_latitude'])
  df_tmp['long_diff'] = np.abs(df_tmp['cust_longitude'] - df_tmp['vendor_longitude'])

  df_tmp['lat_lon_diff'] = np.sqrt((df_tmp['cust_latitude'] - df_tmp['vendor_latitude']) ** 2 + (df_tmp['cust_longitude'] - df_tmp['vendor_longitude']) ** 2)
  df_tmp['vendor_cust_creation_diff'] = df_tmp['created_at_days_since_start_ven'] - df_tmp['created_at_days_since_start_cust']
  df_tmp['vendor_cust_updation_diff'] = df_tmp['updated_at_days_since_start_ven'] - df_tmp['updated_at_days_since_start_cust']

In [22]:
train['target'].value_counts(normalize=True)

0    0.986531
1    0.013469
Name: target, dtype: float64

In [23]:
del orders, vendors, train_locations, test_locations, df_customers, train_customers, test_customers, orders_grouped_by_vendor_id
gc.collect()

305

In [24]:
ID_COL, TARGET_COL = 'CID X LOC_NUM X VENDOR', 'target'
features = [c for c in train.columns if c not in [ID_COL, 'target', 'customer_id', 'dob']]
features = [c for c in features if train[c].isnull().sum()/train.shape[0] < 0.5]

print(f'\nUsing total {len(features)} features')


Using total 102 features


In [25]:
from sklearn.model_selection import GroupKFold
val_df = pd.DataFrame({'customer_id': train['customer_id'].unique(), 'fold': 0})
folds = GroupKFold(7)
for fold_, (_, val_idx) in enumerate(folds.split(val_df['customer_id'], val_df['fold'], val_df['customer_id'])):
  val_df.loc[val_idx, 'fold'] = fold_ + 1
val_df.to_csv('validation_7_folds.csv', index=False)

In [26]:
pd.read_csv('validation_7_folds.csv').head(10)

,customer_id,fold
0,02SFNJH,6
1,RU43CXC,6
2,BDFBPRD,3
3,WMGKW6W,6
4,NDYLK9A,5
5,PB2B28D,2
6,U9YKW1T,1
7,GC6WIE3,2
8,K9BE00O,3
9,AY5F3Q6,5


In [27]:
!ls

 kaggle.json	        train_locations.csv
 orders.csv	        validation_7_folds.csv
 sample_data	        vendors.csv
 SampleSubmission.csv   weekend_food_recommendation
 test_customers.csv    'zindi_weekend_food_recommendation_modelling_v4 (4).csv'
 test_locations.csv     zindiweekendfoodrecommendation.zip
 train_customers.csv
